In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from skimage import io
from skimage.filters import threshold_otsu
from sklearn.linear_model import HuberRegressor

In [2]:
os.getcwd()
data_dir = Path("./Cellpose_output")
csv_suffix = "proce_summary_table.csv"

In [3]:
# Define .csv processing
CHANNEL_MAP = {
    "MeanIntensity_C0": "Mean_DAPI",
    "MeanIntensity_C1": "Mean_Oct4",
    "MeanIntensity_C2": "Mean_GFP",
    "MeanIntensity_C3": "Mean_RFP",
    "MeanIntensity_C6": "Mean_DAPI_mask"
}

cols_to_extract = ["ObjectID", "Centroid_Z", "Centroid_Y", "Centroid_X"] + list(CHANNEL_MAP.keys())

# Load in .csv files
1. Removing last global value row!!
2. Subsetting needed columns
3. Calculating z-corrected intensities

In [ ]:
csv_files = list(data_dir.glob(f"*{csv_suffix}"))
for csv in csv_files:
    prefix = csv.name.replace(csv_suffix, "")
    
    # Read CSV and remove the last global value row
    raw_df = pd.read_csv(csv).iloc[:-1].copy() 
    
    # Extract and rename columns
    selected_cols = [c for c in cols_to_extract if c in raw_df.columns]
    short_df = raw_df[selected_cols].rename(columns=CHANNEL_MAP)

    # Dynamically find all columns that start with "Mean_"
    intensity_cols = [col for col in short_df.columns if col.startswith("Mean_")]
    
    # (Optional) Get channel names without "Mean_" prefix for printing or downstream use
    channel_names = [c.replace("Mean_", "") for c in intensity_cols]

    # Prepare Z values for regression (needs to be 2D array for sklearn)
    # Dropping NaNs in Z just in case Cellpose generated any artifacts
    valid_mask = short_df['Centroid_Z'].notna()
    Z_values = short_df.loc[valid_mask, ['Centroid_Z']].values 
    
    # Initialize Huber Regressor for robust fitting (ignores bright cellular outliers)
    huber = HuberRegressor()

    # Loop through each intensity column to apply Z-correction
    for col in intensity_cols:
        y_values = short_df.loc[valid_mask, col].values
        
        # Fit the model: Intensity vs. Z-depth
        huber.fit(Z_values, y_values)
        
        # Predict the depth-dependent attenuation for all Z positions
        expected_decay = huber.predict(short_df[['Centroid_Z']].fillna(0).values)
        
        # Apply Additive Correction:
        # Subtract the decay trend and add back the intercept (estimated surface intensity)
        corrected_name = f"{col}_zcorr"
        short_df[corrected_name] = short_df[col] - expected_decay + huber.intercept_
        
        # Clip values at 0 to prevent negative intensities
        short_df[corrected_name] = short_df[corrected_name].clip(lower=0)

    print(f"Processed: {csv.name}")
    print(f"Channels corrected: {channel_names}")

    output_path = data_dir / f"{prefix}corrected.csv"
    short_df.to_csv(output_path, index=False)
    print(f"Saved corrected file to {output_path}\n")

Processed: 20260224-E5.5-5-proce_summary_table.csv
Channels corrected: ['DAPI', 'Oct4', 'GFP', 'RFP', 'DAPI_mask']
Saved corrected file to Cellpose_output/20260224-E5.5-5-corrected.csv

Processed: 20260224-E5.5-2-proce_summary_table.csv
Channels corrected: ['DAPI', 'Oct4', 'GFP', 'RFP', 'DAPI_mask']
Saved corrected file to Cellpose_output/20260224-E5.5-2-corrected.csv

Processed: 20260224-E5.5-6-proce_summary_table.csv
Channels corrected: ['DAPI', 'Oct4', 'GFP', 'RFP', 'DAPI_mask']
Saved corrected file to Cellpose_output/20260224-E5.5-6-corrected.csv

Processed: 20260224-E5.5-4-proce_summary_table.csv
Channels corrected: ['DAPI', 'Oct4', 'GFP', 'RFP', 'DAPI_mask']
Saved corrected file to Cellpose_output/20260224-E5.5-4-corrected.csv

Processed: 20260224-E5.5-3-proce_summary_table.csv
Channels corrected: ['DAPI', 'Oct4', 'GFP', 'RFP', 'DAPI_mask']
Saved corrected file to Cellpose_output/20260224-E5.5-3-corrected.csv

